# 8) EE-3.3 Transformers and the Y-bus — Exercises

**Goals:** tap and phase-shift ratios, the branch admittance matrix of a transformer, a Y-bus built from data tables and checked against pandapower, choosing a tap position that keeps every voltage in its band, and steering a flow with a phase shifter.

Per-unit on S_base = 100 MVA; each voltage level uses its nominal voltage as base. The setup cell holds the lecture's five-bus network as tables, the power flow, and pandapower helpers for checking.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import math
import warnings

import numpy as np
import pandas as pd
import pandapower as pp
from scipy.optimize import brentq

warnings.filterwarnings("ignore", category=FutureWarning)
S_BASE = 100.0
OMEGA = 2 * math.pi * 50.0

# the lecture's five-bus network as data tables
BUSES = pd.DataFrame({"vn_kv": [400, 400, 400, 110, 110]})
LINES = pd.DataFrame({"from_bus": [0, 1, 0, 3], "to_bus": [1, 2, 2, 4], "length_km": [150, 120, 200, 40],
                      "r_ohm_per_km": [0.03, 0.03, 0.03, 0.12], "x_ohm_per_km": [0.3, 0.3, 0.3, 0.4],
                      "c_nf_per_km": [12, 12, 12, 9]})
TRAFOS = pd.DataFrame({"hv_bus": [2], "lv_bus": [3], "sn_mva": [250], "vk_percent": [12], "vkr_percent": [0.3],
                       "tap_pos": [-4], "tap_step_percent": [1.25]})
SHUNTS = pd.DataFrame({"bus": [2], "q_mvar": [50]})
LOADS = pd.DataFrame({"bus": [1, 4], "p_mw": [300, 80], "q_mvar": [80, 20]})


def numerical_jacobian(F, x, h=1e-7):
    return np.column_stack([(F(x + e) - F(x - e)) / (2 * h) for e in np.eye(len(x)) * h])


def power_flow_pu(Y, S_inj, tol=1e-12):
    """Newton power flow in per-unit (from the lecture): bus 0 is the slack at 1.0 pu, all others PQ."""
    n = len(Y)
    def F(x):
        V = np.r_[1.0, x[n - 1:]] * np.exp(1j * np.r_[0.0, x[:n - 1]])
        d = V * np.conj(Y @ V) - S_inj
        return np.r_[d.real[1:], d.imag[1:]]
    x = np.r_[np.zeros(n - 1), np.ones(n - 1)]
    for _ in range(40):
        if np.abs(F(x)).max() < tol:
            return np.r_[1.0, x[n - 1:]] * np.exp(1j * np.r_[0.0, x[:n - 1]])
        x = x + np.linalg.solve(numerical_jacobian(F, x), -F(x))
    raise RuntimeError("no convergence")


def pandapower_from_tables(buses, lines, trafos, shunts, loads):
    """The same network in pandapower (S_base = 100 MVA), for checking."""
    net = pp.create_empty_network(sn_mva=S_BASE)
    for v in buses.vn_kv:
        pp.create_bus(net, vn_kv=v)
    pp.create_ext_grid(net, 0, vm_pu=1.0)
    for l in lines.itertuples():
        pp.create_line_from_parameters(net, l.from_bus, l.to_bus, l.length_km, l.r_ohm_per_km, l.x_ohm_per_km,
                                       l.c_nf_per_km, max_i_ka=3)
    for tr in trafos.itertuples():
        pp.create_transformer_from_parameters(net, tr.hv_bus, tr.lv_bus, sn_mva=tr.sn_mva,
                                              vn_hv_kv=buses.vn_kv[tr.hv_bus], vn_lv_kv=buses.vn_kv[tr.lv_bus],
                                              vk_percent=tr.vk_percent, vkr_percent=tr.vkr_percent, pfe_kw=0,
                                              i0_percent=0, tap_side="hv", tap_neutral=0, tap_min=-10, tap_max=10,
                                              tap_step_percent=tr.tap_step_percent, tap_pos=tr.tap_pos,
                                              tap_changer_type="Ratio")
    for sh in shunts.itertuples():
        pp.create_shunt(net, sh.bus, q_mvar=sh.q_mvar)
    for ld in loads.itertuples():
        pp.create_load(net, ld.bus, p_mw=ld.p_mw, q_mvar=ld.q_mvar)
    pp.runpp(net, numba=False)
    return net


def pandapower_pst_flow(shift_deg):
    """Flow (MW) on line A of the lecture's phase-shifter network, computed by pandapower."""
    net = pp.create_empty_network(sn_mva=S_BASE)
    b = [pp.create_bus(net, vn_kv=400) for _ in range(3)]
    pp.create_ext_grid(net, b[0])
    pp.create_line_from_parameters(net, b[0], b[1], 150, 0.03, 0.3, 0, 3)
    pp.create_transformer_from_parameters(net, b[0], b[2], sn_mva=1000, vn_hv_kv=400, vn_lv_kv=400, vk_percent=10,
                                          vkr_percent=0, pfe_kw=0, i0_percent=0, shift_degree=shift_deg)
    pp.create_line_from_parameters(net, b[2], b[1], 150, 0.03, 0.3, 0, 3)
    pp.create_load(net, b[1], p_mw=1000, q_mvar=0)
    pp.runpp(net, numba=False)
    return net.res_line.p_from_mw[0]

### Warm-ups

**1. Tap and phase-shift ratios**

Write `tap_ratio(tap_pos, step_percent, shift_deg=0.0)` returning the complex ratio t = (1 + tap_pos · step/100) · e^{jφ}.

In [2]:
def tap_ratio(tap_pos: int, step_percent: float, shift_deg: float = 0.0) -> complex:
    """Complex off-nominal ratio of a transformer (high-voltage side)."""
    ### BEGIN SOLUTION
    return (1 + tap_pos * step_percent / 100) * np.exp(1j * np.radians(shift_deg))
    ### END SOLUTION

In [3]:
assert np.isclose(tap_ratio(0, 1.25), 1.0)
assert np.isclose(tap_ratio(-4, 1.25), 0.95)
assert np.isclose(abs(tap_ratio(0, 1.25, 30)), 1.0) and np.isclose(np.angle(tap_ratio(0, 1.25, 30), deg=True), 30)

**2. The 2 × 2 branch matrix**

Write `branch_matrix(y, t=1, b_shunt=0)` returning the 2 × 2 admittance matrix of a branch between its two ends (lecture: Y_ff = (y + jb/2)/|t|², Y_fk = −y/t\*, Y_kf = −y/t, Y_kk = y + jb/2). Check three properties.

In [4]:
def branch_matrix(y: complex, t: complex = 1.0, b_shunt: float = 0.0) -> np.ndarray:
    """2x2 admittance matrix of a branch with tap t on side f and total line charging b_shunt."""
    ### BEGIN SOLUTION
    return np.array([[(y + 0.5j * b_shunt) / abs(t) ** 2, -y / np.conj(t)],
                     [-y / t, y + 0.5j * b_shunt]])
    ### END SOLUTION

In [5]:
y = 1 / (0.01 + 0.1j)
assert np.allclose(branch_matrix(y).sum(axis=1), 0), "a plain series branch: rows sum to zero"
M_tap = branch_matrix(y, t=0.95)
assert np.allclose(M_tap, M_tap.T), "a real tap keeps the matrix symmetric"
M_pst = branch_matrix(y, t=tap_ratio(0, 1.25, 10))
assert not np.isclose(M_pst[0, 1], M_pst[1, 0]), "a phase shift breaks the symmetry"
assert np.isclose(branch_matrix(y, b_shunt=0.2)[0, 0], y + 0.1j), "half the line charging at each end"

### Core

**3. Build the Y-bus from the tables**

Write `build_ybus(buses, lines, trafos, shunts)` using `branch_matrix` (lines: per-unit on their own voltage base, with charging b = ωcℓ·Z_base; transformers: z from v_k and v_kr on S_rated changed to S_BASE, tap from `tap_ratio`; shunts: −j·q_mvar/S_BASE). Compare with pandapower's internal Y-bus — for the lecture's tables and for a modified network.

In [6]:
def build_ybus(buses, lines, trafos, shunts, s_base: float = S_BASE) -> np.ndarray:
    """Per-unit Y-bus from pandapower-style tables."""
    ### BEGIN SOLUTION
    Y = np.zeros((len(buses), len(buses)), dtype=complex)
    def add(f, k, M):
        Y[np.ix_([f, k], [f, k])] += M
    for l in lines.itertuples():
        z_base = buses.vn_kv[l.from_bus] ** 2 / s_base
        z = (l.r_ohm_per_km + 1j * l.x_ohm_per_km) * l.length_km / z_base
        add(l.from_bus, l.to_bus, branch_matrix(1 / z, b_shunt=OMEGA * l.c_nf_per_km * 1e-9 * l.length_km * z_base))
    for tr in trafos.itertuples():
        vk, vkr = tr.vk_percent / 100, tr.vkr_percent / 100
        z = (vkr + 1j * math.sqrt(vk**2 - vkr**2)) * s_base / tr.sn_mva
        add(tr.hv_bus, tr.lv_bus, branch_matrix(1 / z, t=tap_ratio(tr.tap_pos, tr.tap_step_percent)))
    for sh in shunts.itertuples():
        Y[sh.bus, sh.bus] += -1j * sh.q_mvar / s_base
    return Y
    ### END SOLUTION

In [7]:
def pandapower_ybus(net):
    order = net._pd2ppc_lookups["bus"]
    return net._ppc["internal"]["Ybus"].toarray()[np.ix_(order, order)]

Y = build_ybus(BUSES, LINES, TRAFOS, SHUNTS)
assert np.abs(Y - pandapower_ybus(pandapower_from_tables(BUSES, LINES, TRAFOS, SHUNTS, LOADS))).max() < 1e-12

trafos_2 = TRAFOS.assign(tap_pos=2)
shunts_2 = pd.concat([SHUNTS, pd.DataFrame({"bus": [4], "q_mvar": [-20]})], ignore_index=True)   # add a capacitor
Y2 = build_ybus(BUSES, LINES, trafos_2, shunts_2)
assert np.abs(Y2 - pandapower_ybus(pandapower_from_tables(BUSES, LINES, trafos_2, shunts_2, LOADS))).max() < 1e-12
changed = np.argwhere(~np.isclose(Y, Y2))
print("entries that changed:", sorted({tuple(int(v) for v in idx) for idx in changed}))

entries that changed: [(2, 2), (2, 3), (3, 2), (4, 4)]


**4. Which tap positions keep every voltage in its band?**

Write `tap_scan(tap_positions)` returning a DataFrame (index: tap position) with every bus voltage (pu) from your own power flow, plus a column `all_in_band` that is `True` when all voltages are within 0.95–1.05 pu.

In [8]:
def tap_scan(tap_positions) -> pd.DataFrame:
    """Bus voltages (pu) of the five-bus network for each tap position, and whether all are in 0.95-1.05 pu."""
    ### BEGIN SOLUTION
    S_inj = np.zeros(len(BUSES), dtype=complex)
    for ld in LOADS.itertuples():
        S_inj[ld.bus] -= (ld.p_mw + 1j * ld.q_mvar) / S_BASE
    rows = {}
    for tap in tap_positions:
        V = power_flow_pu(build_ybus(BUSES, LINES, TRAFOS.assign(tap_pos=tap), SHUNTS), S_inj)
        rows[tap] = {f"bus_{i}": abs(v) for i, v in enumerate(V)}
    table = pd.DataFrame.from_dict(rows, orient="index").rename_axis("tap_pos")
    table["all_in_band"] = table.apply(lambda r: r.between(0.95, 1.05).all(), axis=1)
    return table
    ### END SOLUTION

In [9]:
scan = tap_scan(range(-8, 3))
ok = scan.index[scan.all_in_band].tolist()
assert -4 in ok and 0 not in ok and -6 not in ok
ref = pandapower_from_tables(BUSES, LINES, TRAFOS.assign(tap_pos=ok[0]), SHUNTS, LOADS)
assert np.allclose(scan.loc[ok[0]].drop("all_in_band").astype(float), ref.res_bus.vm_pu, atol=1e-9)
print("tap positions keeping all buses in 0.95-1.05 pu:", ok)
scan.round(4)

tap positions keeping all buses in 0.95-1.05 pu: [-5, -4, -3]


         bus_0   bus_1   bus_2   bus_3   bus_4  all_in_band
tap_pos                                                    
-8         1.0  0.9931  0.9962  1.0936  1.0342        False
-7         1.0  0.9930  0.9961  1.0780  1.0174        False
-6         1.0  0.9930  0.9960  1.0628  1.0010        False
-5         1.0  0.9929  0.9959  1.0479  0.9850         True
-4         1.0  0.9928  0.9958  1.0335  0.9694         True
-3         1.0  0.9928  0.9957  1.0194  0.9540         True
-2         1.0  0.9927  0.9956  1.0056  0.9390        False
-1         1.0  0.9927  0.9955  0.9921  0.9243        False
 0         1.0  0.9926  0.9954  0.9790  0.9099        False
 1         1.0  0.9925  0.9952  0.9662  0.8958        False
 2         1.0  0.9925  0.9951  0.9536  0.8819        False

**5. Interpretation**

Look at buses 3 and 4 in your scan. Why can't the tap fix bus 4 at any voltage you like? What limits the usable range, and which other device from EE-2.2 or EE-3.2 could widen it?

<!-- BEGIN SOLUTION -->
The tap changes the ratio between the 400 kV and 110 kV sides, so it moves **both** 110 kV buses together: bus 3 (the transformer's low-voltage terminal) and bus 4 (the end of the 110 kV line). Because the line between them carries current, bus 4 always sits below bus 3 by the line's voltage drop. Lowering the tap far enough to lift bus 4 to its target pushes bus 3 above 1.05 pu first (tap −6: bus 3 at 1.06), while raising it lets bus 4 fall below 0.95 (tap −2 and above). The usable taps are those where both fit — a narrow window set by the voltage *spread* along the 110 kV line, not by the tap range. Shrinking that spread needs a different tool: a capacitor bank (EE-2.2) at bus 4 supplies the load's reactive power locally, cutting the current and voltage drop on the 110 kV line, after which a wider range of taps keeps both buses in band; at night, when the load is light, a shunt reactor (EE-3.2) prevents the opposite problem.
<!-- END SOLUTION -->

### Challenge

**6. Set a phase shifter to a target flow**

In the lecture's PST network, line A carries 585 MW with no shift. Write `own_pst_flow(shift_deg)` (your own model, `branch_matrix` and `power_flow_pu`, as in the lecture) and `shift_for_target(target_MW)`, which finds the shift angle that brings line A to exactly `target_MW` (use `brentq`). Check it with pandapower, and compare with the DC estimate φ ≈ ΔP · x_loop.

In [10]:
Z_BASE_400 = 400**2 / S_BASE
Z_A = Z_B = (0.03 + 0.3j) * 150 / Z_BASE_400
Z_PST = 0.10j * S_BASE / 1000


def own_pst_flow(shift_deg: float) -> float:
    """Flow (MW) on line A with the PST at shift_deg, from your own model."""
    ### BEGIN SOLUTION
    Y = np.zeros((3, 3), dtype=complex)
    for f, k, z, t in [(0, 1, Z_A, 1.0), (0, 2, Z_PST, tap_ratio(0, 0, shift_deg)), (2, 1, Z_B, 1.0)]:
        Y[np.ix_([f, k], [f, k])] += branch_matrix(1 / z, t=t)
    V = power_flow_pu(Y, np.array([0, -10.0, 0]))
    return float((V[0] * np.conj((V[0] - V[1]) / Z_A)).real * S_BASE)
    ### END SOLUTION


def shift_for_target(target_MW: float) -> float:
    """Phase shift (degrees) that makes line A carry target_MW."""
    ### BEGIN SOLUTION
    return brentq(lambda s: own_pst_flow(s) - target_MW, -30, 30)
    ### END SOLUTION

In [11]:
assert math.isclose(own_pst_flow(0), pandapower_pst_flow(0), rel_tol=1e-8)
phi = shift_for_target(500)
assert math.isclose(pandapower_pst_flow(phi), 500, rel_tol=1e-6), "pandapower confirms 500 MW on line A"
x_loop = (Z_A + Z_B + Z_PST).imag
phi_dc = -math.degrees((own_pst_flow(0) - 500) / S_BASE * x_loop)
assert abs(phi_dc - phi) / abs(phi) < 0.10, "the DC estimate is within 10 %"
print(f"line A to 500 MW: shift {phi:.3f} deg (DC estimate {phi_dc:.3f} deg)")

line A to 500 MW: shift -3.333 deg (DC estimate -3.218 deg)


**7. Interpretation**

A phase shifter can push power away from one path, but not create transmission capacity. Explain why, using what happens to the *other* path in task 6. Why do neighbouring operators coordinate the settings of PSTs on their borders rather than each optimising their own?

<!-- BEGIN SOLUTION -->
A PST only changes how a given transfer **splits** between parallel paths: with the 1,000 MW load fixed, every megawatt removed from line A (585 → 500 MW in task 6) must flow over the PST path instead, so that path's loading rises by the same amount. It relieves a congested line only if the other path has spare capacity; the total that the parallel paths can carry is still limited by their thermal ratings and stability limits. Across borders the same physics creates conflicts: one operator's PST can push loop flows out of its own grid and into a neighbour's, relieving its lines while loading the neighbour's, and two PSTs set by different operators can work against each other. Coordinated settings — agreed in regional security coordination and reflected in flow-based market coupling (Module 04) — use the PSTs to maximise what the whole meshed system can carry safely, instead of shifting congestion back and forth.
<!-- END SOLUTION -->